# Poredjenje Naive Bayes, KNN i CNN modela na zadatku prepoznavanja emocija iz glasa
    Na osnovu evaluacije naših modela za prepoznavanje emocija iz audio signala, može se uočiti jasna razlika u performansama između KNN, Naive Bayes i CNN pristupa. CNN model je pokazao najbolju tačnost i sposobnost generalizacije, posebno zahvaljujući automatskom učenju relevantnih karakteristika iz spektralnih reprezentacija. KNN model je dao solidne rezultate, ali je bio osetljiv na dimenzionalnost i raznolikost podataka, dok je Naive Bayes pokazao najlošije performanse, što je očekivano s obzirom na pretpostavku nezavisnosti karakteristika, koja retko važi za audio signale. Zaključujemo da CNN predstavlja najpouzdanije rešenje za ovaj zadatak, dok su KNN i Naive Bayes jednostavniji modeli sa ograničenom preciznošću.

In [ ]:
# ===============================
# COMPARE MODELS - JUPYTER
# ===============================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
import joblib

# ===============================
# 1️⃣ Učitavanje test podataka
# ===============================
X_test  = np.load("../data/processed_data/cnn/X_test_cnn.npy")
y_test  = np.load("../data/processed_data/cnn/y_test_cnn.npy")

# Ako su podaci 2D, dodaj kanal dimenziju za CNN
X_test_cnn = X_test[:, :, np.newaxis] if X_test.ndim==2 else X_test

# Label encoding i one-hot za CNN
emotion_labels = ["neutral","calm","happy","sad","angry","fearful","disgust","surprised"]
le = LabelEncoder()
le.fit(emotion_labels)
y_test_int = le.transform(y_test)
y_test_enc = to_categorical(y_test_int, num_classes=len(le.classes_))

# ===============================
# 2️⃣ Učitavanje modela
# ===============================
# CNN
cnn_model = load_model("../models/cnn_model.h5")

# KNN i Naive Bayes (pretpostavljamo da su sačuvani sa joblib)
knn_model = joblib.load("../models/knn_model.pkl")
nb_model  = joblib.load("../models/nb_model.pkl")

# Ako je bio scaler korišćen, učitaj ga
scaler = joblib.load("../models/scaler.pkl")
X_test_scaled = scaler.transform(X_test.reshape(-1, X_test.shape[2])).reshape(X_test.shape)

# ===============================
# 3️⃣ Evaluacija i predikcija
# ===============================

# CNN
cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn, y_test_enc, verbose=0)
y_pred_cnn = np.argmax(cnn_model.predict(X_test_cnn), axis=1)

# KNN
y_pred_knn = knn_model.predict(X_test_scaled)
knn_acc = accuracy_score(y_test_int, y_pred_knn)

# Naive Bayes
y_pred_nb = nb_model.predict(X_test_scaled)
nb_acc = accuracy_score(y_test_int, y_pred_nb)

# ===============================
# 4️⃣ Ispis tačnosti
# ===============================
print(f"CNN Accuracy: {cnn_acc*100:.2f}%")
print(f"KNN Accuracy: {knn_acc*100:.2f}%")
print(f"Naive Bayes Accuracy: {nb_acc*100:.2f}%")

# ===============================
# 5️⃣ Konfuzione matrice
# ===============================
models = {
    "CNN": y_pred_cnn,
    "KNN": y_pred_knn,
    "Naive Bayes": y_pred_nb
}

for name, y_pred in models.items():
    cm = confusion_matrix(y_test_int, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=emotion_labels)
    plt.figure(figsize=(10,8))
    disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

# ===============================
# 6️⃣ Bar chart poređenja tačnosti
# ===============================
accuracies = [cnn_acc*100, knn_acc*100, nb_acc*100]
model_names = ["CNN", "KNN", "Naive Bayes"]

plt.figure(figsize=(8,5))
plt.bar(model_names, accuracies, color=['skyblue','salmon','lightgreen'])
plt.ylabel("Accuracy (%)")
plt.title("Comparison of Model Accuracies")
plt.ylim(0, 100)
for i, v in enumerate(accuracies):
    plt.text(i, v+1, f"{v:.2f}%", ha='center')
plt.show()
